In [1]:
from langgraph.graph import StateGraph ,START,END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from  dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage,BaseMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
import operator

In [2]:
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-20b")

In [3]:
class JokeState(TypedDict):
    
    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):
    topic = state['topic']
    prompt = f"generate a joke on the following topic: {state['topic']} make it GenZ style ,based on new current Protect indian topic related (Education)"
    joke = llm.invoke(prompt).content
    return {'joke' : joke}

In [5]:
def generate_explanation(state: JokeState):
    joke  = state['joke']
    prompt = f"Generate a Simple easy to undersatnd explaination of the following joke {state['joke']} "
    explanation  =llm.invoke(prompt).content
    return {'explanation' : explanation}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
initial_state={
    'topic': 'pizza'
}
config1 = {"configurable":{"thread_id":"1"}}
final_response = workflow.invoke(initial_state,config=config1)
print("final_response",final_response)


final_response {'topic': 'pizza', 'joke': '', 'explanation': 'Sure! Please paste the joke you’d like explained, and I’ll give you a simple, easy‑to‑understand breakdown.'}


In [8]:
workflow.invoke(initial_state,config=config1)

{'topic': 'pizza',
 'joke': '**Gen‑Z Pizza‑Edu Punchline**\n\n> The new Protect Indian Education Act just dropped, and even pizza’s like:  \n> “I’m already *crusted* in safety, but if you’re protecting students, you’re basically giving everyone a *cheesy* slice of security. Let’s keep the heat low and the knowledge high! 🍕🛡️📚 #CheeseAndEdu”',
 'explanation': '**What the joke is doing**\n\n1. **It’s about a real thing** – the *Protect Indian Education Act* (a bill that aims to keep Native‑American schools safe and well‑funded).\n\n2. **It’s told in Gen‑Z “pizza” slang** – the speaker is a pizza (or a person who loves pizza) and uses pizza‑related words to talk about the bill.\n\n3. **The wordplay**  \n   * **“I’m already *crusted* in safety”** – “crusted” sounds like “crusted” (the outer crust of a pizza) and also means “already safe” (because the crust protects the inside).  \n   * **“You’re basically giving everyone a *cheesy* slice of security”** – “cheesy” is a pizza adjective, and 

In [9]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '**Gen‑Z Pizza‑Edu Punchline**\n\n> The new Protect Indian Education Act just dropped, and even pizza’s like:  \n> “I’m already *crusted* in safety, but if you’re protecting students, you’re basically giving everyone a *cheesy* slice of security. Let’s keep the heat low and the knowledge high! 🍕🛡️📚 #CheeseAndEdu”', 'explanation': '**What the joke is doing**\n\n1. **It’s about a real thing** – the *Protect Indian Education Act* (a bill that aims to keep Native‑American schools safe and well‑funded).\n\n2. **It’s told in Gen‑Z “pizza” slang** – the speaker is a pizza (or a person who loves pizza) and uses pizza‑related words to talk about the bill.\n\n3. **The wordplay**  \n   * **“I’m already *crusted* in safety”** – “crusted” sounds like “crusted” (the outer crust of a pizza) and also means “already safe” (because the crust protects the inside).  \n   * **“You’re basically giving everyone a *cheesy* slice of security”** – “cheesy” is a p

In [13]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': '**Why did the Indian education minister bring a bowl of pasta to the policy meeting?**  \nBecause the new “Protect\u202fIndian” curriculum says we must **spaghetti‑the syllabus** so it can’t get *cooked up* by foreign trends.  \nNow every student’s learning is *twirled* into knowledge—no cap, it’s totally *lit*!  \n\n🍝📚 #GenZ #PastaPower #ProtectIndianEducation',
 'explanation': '**Joke recap**\n\n> **Why did the Indian education minister bring a bowl of pasta to the policy meeting?**  \n> Because the new “Protect\u202fIndian” curriculum says we must **spaghetti‑the syllabus** so it can’t get *cooked up* by foreign trends.  \n> Now every student’s learning is *twirled* into knowledge—no cap, it’s totally *lit*!\n\n---\n\n### What’s going on?\n\n| Part of the joke | What it means | Why it’s funny |\n|------------------|---------------|----------------|\n| **“Protect\u202fIndian” curriculum** | A made‑up policy that says the education system should stay “pure

In [15]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': '**Gen‑Z Pizza‑Edu Punchline**\n\n> The new Protect Indian Education Act just dropped, and even pizza’s like:  \n> “I’m already *crusted* in safety, but if you’re protecting students, you’re basically giving everyone a *cheesy* slice of security. Let’s keep the heat low and the knowledge high! 🍕🛡️📚 #CheeseAndEdu”', 'explanation': '**What the joke is doing**\n\n1. **It’s about a real thing** – the *Protect Indian Education Act* (a bill that aims to keep Native‑American schools safe and well‑funded).\n\n2. **It’s told in Gen‑Z “pizza” slang** – the speaker is a pizza (or a person who loves pizza) and uses pizza‑related words to talk about the bill.\n\n3. **The wordplay**  \n   * **“I’m already *crusted* in safety”** – “crusted” sounds like “crusted” (the outer crust of a pizza) and also means “already safe” (because the crust protects the inside).  \n   * **“You’re basically giving everyone a *cheesy* slice of security”** – “cheesy” is a pi

In [16]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '**Gen‑Z Pizza‑Edu Punchline**\n\n> The new Protect Indian Education Act just dropped, and even pizza’s like:  \n> “I’m already *crusted* in safety, but if you’re protecting students, you’re basically giving everyone a *cheesy* slice of security. Let’s keep the heat low and the knowledge high! 🍕🛡️📚 #CheeseAndEdu”', 'explanation': '**What the joke is doing**\n\n1. **It’s about a real thing** – the *Protect Indian Education Act* (a bill that aims to keep Native‑American schools safe and well‑funded).\n\n2. **It’s told in Gen‑Z “pizza” slang** – the speaker is a pizza (or a person who loves pizza) and uses pizza‑related words to talk about the bill.\n\n3. **The wordplay**  \n   * **“I’m already *crusted* in safety”** – “crusted” sounds like “crusted” (the outer crust of a pizza) and also means “already safe” (because the crust protects the inside).  \n   * **“You’re basically giving everyone a *cheesy* slice of security”** – “cheesy” is a p

In [17]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f18b14d-0fed-6ebe-8000-b3e0aecb4f37"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f18b14d-0fed-6ebe-8000-b3e0aecb4f37'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-07-29T06:14:54.716669+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b14d-0fe9-66d2-bfff-67c09ce31d21'}}, tasks=(PregelTask(id='ad18ecd3-6bc6-c1ee-f19c-3a4596915da6', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': ''}),), interrupts=())

In [19]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f18b14d-0fed-6ebe-8000-b3e0aecb4f37"}})

{'topic': 'pizza',
 'joke': '**Joke (Gen\u202fZ style, pizza + “Protect\u202fIndian\u202fEducation” vibe)**  \n\n> “Why did the pizza student start a study‑group on campus?  \n> Because the “Protect\u202fIndian\u202fEducation” campaign said you need a *full‑topping* approach to keep the knowledge gap from getting *cheese‑d*.  \n> So every slice is a lesson, every bite is a *grade‑A* meme. 🍕📚 #WokeBites #EduPizza”',
 'explanation': '**What the joke is doing**\n\n1. **Setting the scene**  \n   *A “pizza student”* is a student who loves pizza (or who is a “pizza” in the slang sense of “awesome”).  \n   The question asks why that student would start a study‑group on campus.\n\n2. **The punch‑line reference**  \n   *“Protect Indian Education”* is a real‑world campaign that urges people to help keep schools in India strong.  \n   The joke says the campaign told the student that you need a **full‑topping** approach—meaning you have to use every resource and idea (like putting every topping on

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': '**Joke (Gen\u202fZ style, pizza + “Protect\u202fIndian\u202fEducation” vibe)**  \n\n> “Why did the pizza student start a study‑group on campus?  \n> Because the “Protect\u202fIndian\u202fEducation” campaign said you need a *full‑topping* approach to keep the knowledge gap from getting *cheese‑d*.  \n> So every slice is a lesson, every bite is a *grade‑A* meme. 🍕📚 #WokeBites #EduPizza”', 'explanation': '**What the joke is doing**\n\n1. **Setting the scene**  \n   *A “pizza student”* is a student who loves pizza (or who is a “pizza” in the slang sense of “awesome”).  \n   The question asks why that student would start a study‑group on campus.\n\n2. **The punch‑line reference**  \n   *“Protect Indian Education”* is a real‑world campaign that urges people to help keep schools in India strong.  \n   The joke says the campaign told the student that you need a **full‑topping** approach—meaning you have to use every resource and idea (like putt

In [22]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f18b165-7bd8-650e-8000-d4f88df735a2'}}

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18b165-7bd8-650e-8000-d4f88df735a2'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-07-29T06:25:50.277531+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='9cb21f89-3045-2acc-ee01-587191f21382', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': '**Joke (Gen\u202fZ style, pizza + “Protect\u202fIndian\u202fEducation” vibe)**  \n\n> “Why did the pizza student start a study‑group on campus?  \n> Because the “Protect\u202fIndian\u202fEducation” campaign said you need a *full‑topping* approach to keep the knowledge gap from getting *cheese‑d*.  \n> So every slice is a lesson,

In [24]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f18b165-7bd8-650e-8000-d4f88df735a2"}})

{'topic': 'samosa',
 'joke': '**Gen‑Z‑style samosa joke on Protect Indian Education**\n\n> Why did the samosa enroll in the “Protect Indian Education” program?  \n>   \n> Because it heard the new bill was all about **“fold, fill, and test”**—the only snack that can *crunch* knowledge without crumbing the future.  \n>   \n> #SamosaSavesEdu #LitLearning #OnFleek 📚🥟',
 'explanation': '**What the joke is doing (in plain English)**  \n\n| Part of the joke | Why it’s funny / what it means |\n|------------------|--------------------------------|\n| **“Why did the samosa enroll in the ‘Protect Indian Education’ program?”** | A samosa is a tasty, folded snack. Saying it “enrolls” in a serious education‑protection bill is absurd – it’s like a potato saying it wants to go to college. The absurdity is the first laugh. |\n| **“Because it heard the new bill was all about ‘fold, fill, and test’”** | The bill’s name (Protect Indian Education) is a mouth‑feel of a cooking instruction. “Fold, fill, and 

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': '**Gen‑Z‑style samosa joke on Protect Indian Education**\n\n> Why did the samosa enroll in the “Protect Indian Education” program?  \n>   \n> Because it heard the new bill was all about **“fold, fill, and test”**—the only snack that can *crunch* knowledge without crumbing the future.  \n>   \n> #SamosaSavesEdu #LitLearning #OnFleek 📚🥟', 'explanation': '**What the joke is doing (in plain English)**  \n\n| Part of the joke | Why it’s funny / what it means |\n|------------------|--------------------------------|\n| **“Why did the samosa enroll in the ‘Protect Indian Education’ program?”** | A samosa is a tasty, folded snack. Saying it “enrolls” in a serious education‑protection bill is absurd – it’s like a potato saying it wants to go to college. The absurdity is the first laugh. |\n| **“Because it heard the new bill was all about ‘fold, fill, and test’”** | The bill’s name (Protect Indian Education) is a mouth‑feel of a cooking instructio